In [2]:
"""
Gold Layer: builds Fact_Trips + dimension tables as managed Hive tables.
Run from Jupyter/PySpark shell (Hive-enabled SparkSession, as in Lab 2).

Reads:
  hdfs:///tlc/staging/fact_trip_clean   (Member 2's ELT output)
  <zones source>                        (attribute table, 265 rows)
  <geospatial output>                   (Member 2/3 geohash output, if available)

Writes (managed Hive tables, Parquet + snappy):
  {HIVE_DB}.dim_datetime
  {HIVE_DB}.dim_location
  {HIVE_DB}.dim_trip_flags
  {HIVE_DB}.dim_dispatch
  {HIVE_DB}.Fact_Trips   (partitioned by year, month, day — matches silver layer)
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.enableHiveSupport().getOrCreate()

# ---- Config: adjust these two paths/names for your environment ----
HIVE_DB = "uber_dw"
SILVER_PATH = "hdfs:///gold_layer/data_source"
# ZONES_PATH = "hdfs:///tlc/raw/zones"        # the 265-row attribute table
# GEO_PATH = "hdfs:///tlc/staging/staging_rides_geo"  # uncomment once Member 2 delivers this

spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")

2026-09-03 22:40:52,628 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 22:40:52,629 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-09-03 22:40:54,230 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
2026-09-03 22:40:54,243 ERROR metastore.RetryingHMSHandler: AlreadyExistsException(message:Database uber_dw already exists)
	at org.apache.hadoop.hive.metastore.HiveMetaStore$HMSHandler.create_database(HiveMetaStore.java:925)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invokeInternal(RetryingHMSHandler.java:148)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invoke(Re

DataFrame[]

In [3]:
# =====================================================================
# 1. Read sources
# =====================================================================
trips = spark.read.parquet(SILVER_PATH)

In [4]:
# =====================================================================
# 2. dim_location  — Extract directly from staging data using union
# =====================================================================

# 1. Grab all the pickup locations and rename the columns
pu_locations = trips.select(
    F.col("pu_location_id").alias("location_id"),
    F.col("pu_borough").alias("borough"),
    F.col("pu_zone").alias("zone"),
    F.col("pu_service_zone").alias("service_zone")
)

# 2. Grab all the dropoff locations and rename the columns
do_locations = trips.select(
    F.col("do_location_id").alias("location_id"),
    F.col("do_borough").alias("borough"),
    F.col("do_zone").alias("zone"),
    F.col("do_service_zone").alias("service_zone")
)

# 3. Stack them together, remove duplicates, and add the empty geo_hash
dim_location = (
    pu_locations.union(do_locations)
    .dropDuplicates(["location_id"])
    .filter(F.col("location_id").isNotNull()) 
    .withColumn("geo_hash", F.lit(None).cast("string"))
)

dim_location.show(5)

+-----------+---------+--------------------+------------+--------+
|location_id|  borough|                zone|service_zone|geo_hash|
+-----------+---------+--------------------+------------+--------+
|        148|Manhattan|     Lower East Side| Yellow Zone|    null|
|        243|Manhattan|Washington Height...|   Boro Zone|    null|
|         31|    Bronx|          Bronx Park|   Boro Zone|    null|
|         85| Brooklyn|             Erasmus|   Boro Zone|    null|
|        137|Manhattan|            Kips Bay| Yellow Zone|    null|
+-----------+---------+--------------------+------------+--------+
only showing top 5 rows



In [5]:
# =====================================================================
# 3. dim_datetime — merged date+hour grain, generated calendar range
#    (not just observed dates, so future months join without a reload)
# =====================================================================
calendar = (
    spark.sql("SELECT sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day) AS d")
    .withColumn("full_date", F.explode("d"))
    .drop("d")
    .withColumn("hour", F.explode(F.sequence(F.lit(0), F.lit(23))))
)

dim_datetime = (
    calendar
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))  # 1=Sun..7=Sat
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7))
    .withColumn("is_peak_hour", F.col("hour").isin(7, 8, 9, 16, 17, 18, 19))  # adjust to your definition
    .withColumn(
        "datetime_key",
        (F.date_format("full_date", "yyyyMMdd").cast("long") * 100) + F.col("hour")
    )
    .select("datetime_key", "full_date", "year", "month", "day", "day_of_week", "hour",
            "is_weekend", "is_peak_hour")
)
dim_datetime.show(5)

+------------+----------+----+-----+---+-----------+----+----------+------------+
|datetime_key| full_date|year|month|day|day_of_week|hour|is_weekend|is_peak_hour|
+------------+----------+----+-----+---+-----------+----+----------+------------+
|  2020010100|2020-01-01|2020|    1|  1|          4|   0|     false|       false|
|  2020010101|2020-01-01|2020|    1|  1|          4|   1|     false|       false|
|  2020010102|2020-01-01|2020|    1|  1|          4|   2|     false|       false|
|  2020010103|2020-01-01|2020|    1|  1|          4|   3|     false|       false|
|  2020010104|2020-01-01|2020|    1|  1|          4|   4|     false|       false|
+------------+----------+----+-----+---+-----------+----+----------+------------+
only showing top 5 rows



In [6]:
# =====================================================================
# 4. dim_trip_flags — junk dimension, distinct flag combinations
#    (includes was_on_scene_matched, confirmed present in silver schema)
# =====================================================================
flag_cols = ["shared_request_flag", "shared_match_flag", "access_a_ride_flag",
             "wav_request_flag", "wav_match_flag", "was_on_scene_matched"]

dim_trip_flags = (
    trips.select(*flag_cols).dropDuplicates()
    .withColumn("flag_key", F.row_number().over(Window.orderBy(*flag_cols)))
)
dim_trip_flags.show(5)

2026-09-03 22:41:03,932 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+-----------------+------------------+----------------+--------------+--------------------+--------+
|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|was_on_scene_matched|flag_key|
+-------------------+-----------------+------------------+----------------+--------------+--------------------+--------+
|              false|            false|             false|           false|         false|               false|       1|
|              false|            false|             false|           false|         false|                true|       2|
|              false|            false|             false|           false|          true|               false|       3|
|              false|            false|             false|           false|          true|                true|       4|
|              false|            false|             false|            true|          true|                true|       5|
+-------------------+-----------

In [7]:
# =====================================================================
# 5. dim_dispatch — distinct license/base combinations
#    (dispatching_base_num added back in, per silver schema confirmation)
# =====================================================================
dispatch_cols = ["license_num", "dispatching_base_num", "originating_base_num"]

dim_dispatch = (
    trips.select(*dispatch_cols).dropDuplicates()
    .withColumn("dispatch_base_id", F.row_number().over(Window.orderBy(*dispatch_cols)))
)
dim_dispatch.show(5)

2026-09-03 22:41:05,104 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+--------------------+--------------------+----------------+
|license_num|dispatching_base_num|originating_base_num|dispatch_base_id|
+-----------+--------------------+--------------------+----------------+
|     HV0003|              B03404|              B03404|               1|
|     HV0005|              B03406|              B03406|               2|
|     HV0005|              B03406|             UNKNOWN|               3|
+-----------+--------------------+--------------------+----------------+



In [8]:
fact_trips = (
    trips
    .withColumn(
        "trip_id",
        F.sha2(F.concat_ws("||", "license_num", "pickup_datetime", 
                           "pu_location_id", "do_location_id"), 256)
    )
    .withColumn(
        "pickup_datetime_key",
        (F.date_format("pickup_datetime", "yyyyMMdd").cast("long") * 100) + F.col("pickup_hour")
    )
    .withColumn(
        "dropoff_datetime_key",
        (F.date_format("dropoff_datetime", "yyyyMMdd").cast("long") * 100) + F.col("dropoff_hour")
    )
    .withColumn("year", F.year("pickup_datetime"))
    .withColumn("month", F.month("pickup_datetime"))
    .withColumn("day", F.dayofmonth("pickup_datetime"))
    # ------------------------------------------
    .join(dim_trip_flags, on=flag_cols, how="left")
    .join(dim_dispatch, on=dispatch_cols, how="left")
    .select(
        "trip_id",
        "pickup_datetime_key", "dropoff_datetime_key",
        F.col("pu_location_id").alias("pickup_location_id"),
        F.col("do_location_id").alias("dropoff_location_id"),
        "dispatch_base_id", "flag_key",
        "trip_miles", "trip_time",
        "base_passenger_fare", "tolls", "bcf", "sales_tax",
        "congestion_surcharge", "airport_fee", "tips", "driver_pay",
        "total_fare", F.col("trip_duration_sec").alias("total_duration_sec"),
        "year", "month", "day",  # Now it will find these successfully!
    )
)
fact_trips.show(5)

2026-09-03 22:41:06,323 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 22:41:06,330 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 22:41:06,345 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-------------------+--------------------+------------------+-------------------+----------------+--------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+------------------+------------------+----+-----+---+
|             trip_id|pickup_datetime_key|dropoff_datetime_key|pickup_location_id|dropoff_location_id|dispatch_base_id|flag_key|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|        total_fare|total_duration_sec|year|month|day|
+--------------------+-------------------+--------------------+------------------+-------------------+----------------+--------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+------------------+------------------+----+-----+---+
|5776cb59097a4a43c...|         2024010109|          2024010109|               144|                114|               1|  

In [9]:
# =====================================================================
# 7. Referential integrity checks — run on the FULL dataset, not the
#    DEV_MODE 500k sample (that sample only covered Jan 1-2 and can't
#    validate this).
# =====================================================================
orphan_pickup = fact_trips.join(dim_location, fact_trips.pickup_location_id == dim_location.location_id, "left_anti").count()
orphan_dropoff = fact_trips.join(dim_location, fact_trips.dropoff_location_id == dim_location.location_id, "left_anti").count()
orphan_datetime = fact_trips.join(dim_datetime, fact_trips.pickup_datetime_key == dim_datetime.datetime_key, "left_anti").count()
print(f"Orphan pickup locations: {orphan_pickup}")
print(f"Orphan dropoff locations: {orphan_dropoff}")
print(f"Orphan pickup datetimes: {orphan_datetime}")

Orphan pickup locations: 0
Orphan dropoff locations: 0
Orphan pickup datetimes: 0


In [10]:
# =====================================================================
# 8. Write as EXTERNAL Hive tables
# =====================================================================
# Define where the actual Parquet files will sit in HDFS
EXTERNAL_BASE_PATH = "hdfs:///gold_layer/data_source"

for name, df in [
    ("dim_datetime", dim_datetime),
    ("dim_location", dim_location),
    ("dim_trip_flags", dim_trip_flags),
    ("dim_dispatch", dim_dispatch),
]:
    df.write \
        .mode("overwrite") \
        .format("parquet") \
        .option("compression", "snappy") \
        .option("path", f"{EXTERNAL_BASE_PATH}/{name}") \
        .saveAsTable(f"{HIVE_DB}.{name}")

# Write the Fact table as EXTERNAL and Partitioned
fact_trips.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .partitionBy("year", "month", "day") \
    .option("path", f"{EXTERNAL_BASE_PATH}/Fact_Trips") \
    .saveAsTable(f"{HIVE_DB}.Fact_Trips")

print("Gold layer build complete. EXTERNAL Tables available in Hive database:", HIVE_DB)

2026-09-03 22:41:14,748 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-09-03 22:41:14,787 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-09-03 22:41:14,788 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 22:41:14,788 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-09-03 22:41:24,130 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 22:41:25,329 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 22:41:26,635 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause

Gold layer build complete. EXTERNAL Tables available in Hive database: uber_dw
